# 10. Komunikace s databázovým systémem - Připojení, Ukládání a načítání dat, Mapování entit v OOP

### Připojení k RDBMS
* Připojení se realizuje přes knihovnu dodávanou přímo výrobcem databáze
* Přihlašovací údaje se do zdrojového kódu nepíší, načítají se dynamicky z externích konfiguračních souborů
* Vužívají se vytvářející návrhové vzory (např. Singleton, Lazy initialization)

### Operace C-R-U-D
* Veškerá manipulace nad tabulkou odpovídá základním operacím Create (příkaz `INSERT`), Read (`SELECT`), Update (`UPDATE`) a Delete (`DELETE`)
* Využívá se strukturální vzor Facade, hlavnímu programu nabídne pouze čisté a jednoduché rozhraní a SQL logiku vykoná skrytě
* Dalším přístupem je třívrstvá architektura (3-Tier)
* Při ukládání a načítání hrozí riziko SQL injection, vzniká nevalidovaným vstupem od uživatele
* Prepared statements a parametrizované dotazy tomu zabranují

### Mapování entit v OOP
* Objektově relační mapování (ORM) je k zrcadlení databázových řádků do objektových tříd
* Slouží k tomu aby vývojář nemusel psát manuálně SQL příkazy
* ORM může generovat více databázových dotazů než klasické ruční SQL
* Bez ORM lze mapování vyřešit návrhovými vzory
    * DAO (Data Access Object) / Table Gateway - pro jednu tabulku v DB existuje v programu přesně jedna třída, disponuje metodami jako `findByName()` nebo `getAll()` a veškerými CRUD
    * Active Record a Row Gateway - každý řádek z tabulky je v operační paměti reprezentován objektem,u sebe drží atributy a zároveň má metodu `save()`, kterou se zaktualizuje či vloží do DB

In [2]:
user = "SYSTEM"
password = "student"
dsn = "localhost:1521/XE"
encoding = "UTF-8"

In [3]:
import cx_Oracle

class Student:
    def __init__(self, id, jmeno, prijmeni):
        self.id = id
        self.jmeno = jmeno
        self.prijmeni = prijmeni

    def __str__(self):
        return f"Student {self.jmeno} {self.prijmeni}"

class StudentDAO:
    def __init__(self, connection):
        self.connection = connection

    def create(self, student: Student):
        cursor = self.connection.cursor()
        cursor.execute(
            "INSERT INTO STUDENT (ID, NAME, SURNAME) VALUES (:id, :jmeno, :prijmeni)",
            {"id":student.id, "jmeno": student.jmeno, "prijmeni": student.prijmeni}
        )
        cursor.close()
        self.connection.commit()
        return "Student ulozen do DB"

    def read(self, jmeno:str):
        cursor = self.connection.cursor()
        cursor.execute("SELECT * FROM STUDENT WHERE NAME = :jmeno", {"jmeno": jmeno})

        vysledek = cursor.fetchone()
        student = Student(vysledek[0], vysledek[1], vysledek[2])
        cursor.close()
        return student

if __name__ == "__main__":
    connection = cx_Oracle.connect(user=user, password=password, dsn=dsn, encoding=encoding)

    connection.cursor().execute("CREATE TABLE STUDENT (ID NUMBER PRIMARY KEY, NAME VARCHAR2(50), SURNAME VARCHAR2(50))")

    student = Student(1,"Jan", "Novak")
    studentDAO = StudentDAO(connection)
    studentDAO.create(student)
    print(studentDAO.read("Jan"))

DatabaseError: DPI-1047: Cannot locate a 64-bit Oracle Client library: "The specified module could not be found". See https://cx-oracle.readthedocs.io/en/latest/user_guide/installation.html for help